# agent_framework_mistral

## Example 1: the chat client reports the model and server it was given

In [ ]:
# Example 1: MistralChatClient with an explicit model, key and server URL
from agent_framework_mistral import MistralChatClient

client = MistralChatClient(model="mistral-small-latest", api_key="test-key", server_url="http://127.0.0.1:9")
assert client.model == "mistral-small-latest", client.model
print("Example 1:", type(client).__name__, client.model)

## Example 2: settings from MISTRAL_* environment variables

In [ ]:
# Example 2: with no arguments the client reads its key and model from the environment
import os
from unittest import mock
from agent_framework_mistral import MistralChatClient

with mock.patch.dict(os.environ, {"MISTRAL_API_KEY": "env-key", "MISTRAL_CHAT_MODEL": "mistral-large-latest"}):
    client = MistralChatClient()
assert client.model == "mistral-large-latest", client.model
print("Example 2: model from the environment:", client.model)

## Example 3: a request to a closed port fails cleanly

In [ ]:
# Example 3: nothing listens on the port, so get_response raises the framework's ChatClientException
import socket
from agent_framework import Message
from agent_framework.exceptions import ChatClientException
from agent_framework_mistral import MistralChatClient

with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0)); port = probe.getsockname()[1]
client = MistralChatClient(model="mistral-small-latest", api_key="k", server_url=f"http://127.0.0.1:{port}")
try:
    await client.get_response([Message(role="user", contents=["bonjour"])])
except ChatClientException as error:
    print("Example 3: refused as expected:", str(error)[:80])
else:
    raise AssertionError("a request to a closed port succeeded")

## Example 4: the embedding client

In [ ]:
# Example 4: MistralEmbeddingClient is built the same way and reports its embedding model
from agent_framework_mistral import MistralEmbeddingClient

embedder = MistralEmbeddingClient(model="mistral-embed", api_key="test-key", server_url="http://127.0.0.1:9")
assert embedder.model == "mistral-embed", embedder.model
print("Example 4:", type(embedder).__name__, embedder.model)

## Example 5: `agent_framework.mistral` re-exports this package

In [ ]:
# Example 5: the umbrella namespace hands back the same classes
import importlib.metadata
import agent_framework.mistral as via_core
import agent_framework_mistral as direct

assert via_core.MistralChatClient is direct.MistralChatClient
print("Example 5:", importlib.metadata.version("agent-framework-mistral"), "re-exported through agent_framework.mistral")